In [ ]:
"""総合演習2 社会系シミュレーション 第3回 (2026-11-25)

複数のボールが2次元領域を等速直線運動するシミュレーション (壁あり)
"""

import random
from typing import Final

import matplotlib.pyplot as plt
from IPython.display import HTML, display
from matplotlib import animation
from matplotlib.artist import Artist


# ========== パラメータ ==========
NUM_AGENTS: Final[int] = 10       # エージェントの数
X_MIN: Final[float] = 0           # x座標の最小値
X_MAX: Final[float] = 10          # x座標の最大値
Y_MIN: Final[float] = 0           # y座標の最小値
Y_MAX: Final[float] = 10          # y座標の最大値
VX_ABS: Final[float] = 1          # x方向の速度の絶対値の上限
VY_ABS: Final[float] = 1          # y方向の速度の絶対値の上限

DT: Final[float] = 0.01           # 時間刻み幅
NUM_STEPS: Final[int] = 2000      # 時間ステップ数
PLOT_INTERVAL: Final[int] = 50    # 可視化するステップ間隔
# ==============================


LX: Final[float] = X_MAX - X_MIN
LY: Final[float] = Y_MAX - Y_MIN
MARGIN: Final[float] = 0.1


class MovableAgent:
    """可動エージェントを表すクラス

    Attributes
    ----------
    x : float
        エージェントのx座標
    y : float
        エージェントのy座標
    vx : float
        エージェントのx方向の速度
    vy : float
        エージェントのy方向の速度
    """

    def __init__(self, init_x: float, init_y: float, init_vx: float, init_vy: float) -> None:
        """MovableAgentクラスのインスタンスのイニシャライザ

        Parameters
        ----------
        init_x : float
            エージェントの初期x座標
        init_y : float
            エージェントの初期y座標
        init_vx : float
            エージェントのx方向の初期速度
        init_vy : float
            エージェントのy方向の初期速度
        """

        self.x: float = init_x
        self.y: float = init_y
        self.vx: float = init_vx
        self.vy: float = init_vy

    def move(self) -> None:
        """エージェントを1ステップ移動させる

        Notes
        -----
        Euler法を使用
        """

        self.x += self.vx * DT
        self.y += self.vy * DT

        # 壁に衝突したら速度を反転
        if not (X_MIN <= self.x <= X_MAX):
            self.vx *= -1
        if not (Y_MIN <= self.y <= Y_MAX):
            self.vy *= -1


class MovableAgentSystem:
    """可動エージェント全体を表すクラス

    Attributes
    ----------
    num_agents : int
        エージェントの数
    agents : list[MovableAgent]
        全ての可動エージェントを格納するリスト
    time : float
        現在の時刻
    """

    def __init__(self, num_agents: int) -> None:
        """MovableAgentSystemクラスのインスタンスのイニシャライザ

        Parameters
        ----------
        num_agents : int
            エージェントの数

        Notes
        -----
        エージェントの初期位置と初期速度はランダムに設定
        """

        self.num_agents: int = num_agents
        self.agents: list[MovableAgent] = []
        self.time: float = 0

        for _ in range(num_agents):
            init_x: float = X_MIN + (X_MAX - X_MIN) * random.random()
            init_y: float = Y_MIN + (Y_MAX - Y_MIN) * random.random()
            init_vx: float = VX_ABS * 2*(random.random() - 0.5)
            init_vy: float = VY_ABS * 2*(random.random() - 0.5)

            self.agents.append(MovableAgent(init_x, init_y, init_vx, init_vy))

    def time_evolution(self) -> None:
        """システムを1ステップ時間発展させる"""

        for agent in self.agents:
            agent.move()

        self.time += DT

    def visualize(self) -> list[Artist]:
        """各タイムステップでのエージェントの位置を可視化する

        Returns
        -------
        frame : list[Artist]
            可視化結果のArtistオブジェクトのリスト

        Notes
        -----
        matplotlib.animation.ArtistAnimationを使用
        """

        frame: list[Artist] = [
            ax.text(X_MIN, Y_MAX+1.5*MARGIN*LY, f't = {self.time:.2f}')
        ]
        for agent in self.agents:
            frame += ax.plot(agent.x, agent.y, color="blue", marker="o")

        return frame


if __name__ == '__main__':

    # 可動エージェントを初期化
    system: MovableAgentSystem = MovableAgentSystem(NUM_AGENTS)

    # 可視化の準備
    fig, ax = plt.subplots(figsize=(5, 5), dpi=150)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_xlim(X_MIN-MARGIN*LX, X_MAX+MARGIN*LX)
    ax.set_ylim(Y_MIN-MARGIN*LY, Y_MAX+MARGIN*LY)
    ax.grid()
    ax.set_aspect('equal')

    # 壁の描画
    ax.axvspan(X_MIN-MARGIN*LX, X_MIN, color='gray')
    ax.axvspan(X_MAX, X_MAX+MARGIN*LX, color='gray')
    ax.axhspan(Y_MIN-MARGIN*LY, Y_MIN, color='gray')
    ax.axhspan(Y_MAX, Y_MAX+MARGIN*LY, color='gray')

    # 初期時刻の可視化
    artists: list[list[Artist]] = []
    artists.append(system.visualize())

    # 時間発展
    for it in range(NUM_STEPS):
        system.time_evolution()

        if (it+1) % PLOT_INTERVAL == 0:
            artists.append(system.visualize())

    # アニメーションの作成
    anim = animation.ArtistAnimation(fig, artists)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))
    anim.save('./social_3rd_ball2d_wall.gif', writer="pillow")